# M5.S5 · Guided integration lab
## Vector search: correct, parallel and reproducible

**About 45–60 minutes of active guided work; queue wait is additional.** This is Part 1 of M5S5.
The separate individual cluster assignment and group milestone have their own instructions.
The module quiz will be provided separately.

A retrieval service compares a query with stored vectors and returns the closest item.
We implement **exact top-1 cosine search in C**. The deterministic vectors represent an embedding
collection, but are synthetic: we do not load documents, train an embedding model, or evaluate semantic relevance.
This isolates the systems problem without downloads or package installations.

By the end you should be able to:
1. Check an exact search result against an independent small reference.
2. Explain MPI shards, OpenMP local work and a global result merge (M5S1–S2).
3. Run bounded comparisons through Slurm and interpret actual timings.
4. Explain what an existing Apptainer image can preserve and what remains host-dependent (M5S3).
5. Keep the code, configuration, evidence and limitations needed for a project handover (M5S4).

**Route:** tiny correctness check → OpenMP measurements → hybrid design → container rerun → evidence review.
Native CPU execution and the Apptainer result-validation workflow are core practical activities. Hybrid execution requires a working site MPI environment; a broken or unavailable route must be reported honestly.
No GPU is required. No successful cluster run is assumed.


## 0 · Run on SciTech

Open https://jupyter.iesci.tech:8085/ and choose **File → Open from URL**. Paste the raw notebook URL
provided with this session, choose Python 3 and save your own copy. This notebook writes into a new
student-owned directory and contains its own C source, runner and batch scripts.

The Python cells prepare files, check a tiny fixture and analyse reports. **Significant C workloads run
in separate Slurm allocations**, not on the login node or an ordinary Jupyter kernel.
If `sbatch` is unavailable in Jupyter, use the terminal instructions below in your cluster home directory.
The files must be on storage visible to both the notebook and compute nodes.

Use the [course cluster guide](https://github.com/OscarDiez/hpc_course/blob/main/cluster/README.md).
From outside campus: `ssh -J <username>@ssh.iesci.tech <username>@10.205.20.10`.
On the login node inspect `hostname`, `sinfo`, `module avail`, `module list`, `which gcc` and `which mpicc`.
The generated **site_env.sh** already contains the configuration used in the successful SciTech instructor preflight (job 24943): system compiler/MPI selection and the shared Python image. Each batch job sources this file. Inspect it and record the actual executable paths; no extra compiler module is required for this tested route. Do not mix in an unrelated EESSI compiler. A running notebook kernel keeps its own interpreter.
No account/QoS is added. The scripts request one node, four CPU cores total, 1 GB and six minutes.

PASS = an executed check passed; FAIL = an attempted check failed; SKIPPED = an optional feature is
disabled/unavailable; NOT RUN = no execution evidence yet. A missing optional feature is not a benchmark result.

**Default execution:** running the submission cell submits three bounded Slurm jobs automatically. Their IDs are saved in the work directory and reused on reruns, including kernel restarts from the same notebook directory. CPU runs first; hybrid follows CPU success; container replay follows CPU success and hybrid termination. The submission cell waits up to 45 seconds, including queue time. If jobs are still queued/running, rerun the submission/status cell and the result cells later; no duplicate jobs are submitted. All significant computation stays in Slurm.


In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, shutil, struct, subprocess, sys, tempfile
from statistics import median
if 'WORK' not in globals():
    WORK = (Path.cwd()/'m5s5_vector_lab').resolve()
    WORK.mkdir(parents=True, exist_ok=True)
STATUS = {}
print('Working directory:', WORK)
print('Python:', sys.version.split()[0])
print('gcc:', shutil.which('gcc'), 'sbatch:', shutil.which('sbatch'))
print('Cluster measurements: NOT RUN until a batch report exists.')
def run(args, env=None, timeout=30):
    p = subprocess.run([str(a) for a in args], cwd=WORK, text=True,
                       capture_output=True, env=env, timeout=timeout)
    if p.returncode:
        raise RuntimeError(f'FAIL: command {args!r}, exit {p.returncode}\n{p.stdout}\n{p.stderr}')
    return p.stdout


## Environment inspection · before changing software
Compare the current Jupyter interpreter with the compiler/MPI paths in a shell. A module loaded in
another terminal does not replace an already running notebook kernel. Record `module list`,
`type -a gcc mpicc python3 apptainer`, and `mpicc --showme` when MPI exists. Use the supplied tested configuration in `site_env.sh` for batch jobs; do not repair a mixed stack by adding random library paths. The initial probe describes the current shell, which can differ from the configured batch environment. The environment probe below is read-only and may show unavailable commands.


In [ ]:
ENV_PROBE = '#!/bin/bash -l\n# Read-only environment inspection; no jobs or packages are installed here.\necho "HOST=$(hostname)"\nmodule -t list 2>&1 || true\ntype -a gcc mpicc python3 apptainer 2>&1 || true\ngcc --version 2>&1 | head -1 || true\nmpicc --showme 2>&1 || true\napptainer --version 2>&1 || true\n# Compare compiler/MPI wrapper paths; do not mix system MPI and another stack.\n'
(WORK/'environment_probe.sh').write_text(ENV_PROBE)
print('Current notebook interpreter:', sys.executable)
print(run(['bash', '-l', 'environment_probe.sh']))
print('Record these initial paths. Batch jobs source the supplied tested site_env.sh separately.')


## E1 · What does “nearest” mean?

**Real problem:** the service must return the correct item before we care how fast it runs.
**Prediction:** if each vector has unit length, how do we rank cosine similarity?
**Code:** cosine is the dot product of the two normalized vectors. We use float storage and double
accumulation. The highest score wins; an exactly equal computed score uses the smaller global ID.
IDs are zero-based. The reference uses the same generator and arithmetic precision, implemented independently in Python.

The next cell writes the complete C program. Inspect `vector`, the row-scoring loop and the result merge.
`mix` generates each component from its global row ID, so MPI partitions do not change the input.
This is deterministic data generation, not a scientific random-number generator.


In [ ]:
C_SOURCE = r"""/* M5S5 guided exact top-1 search. Deterministic synthetic vectors, not text embeddings.
 * Serial: gcc -O3 -std=c11 search.c -lm -o search_serial
 * OpenMP: gcc -O3 -std=c11 -fopenmp search.c -lm -o search_omp
 * Hybrid: mpicc -O3 -std=c11 -fopenmp -DUSE_MPI search.c -lm -o search_mpi
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <math.h>
#include <time.h>
#ifdef _OPENMP
#include <omp.h>
#endif
#ifdef USE_MPI
#include <mpi.h>
#endif
static double now(void) { struct timespec t; clock_gettime(CLOCK_MONOTONIC,&t); return t.tv_sec+t.tv_nsec*1e-9; }
static void stop(const char *why) {
 fprintf(stderr,"FAIL: %s\n",why);
#ifdef USE_MPI
 MPI_Abort(MPI_COMM_WORLD,2);
#endif
 exit(2);
}
/* Stateless 32-bit hash: each rank can generate its own rows by global ID. */
static uint32_t mix(uint32_t x) { x^=x>>16; x*=0x7feb352dU; x^=x>>15; x*=0x846ca68bU; return x^(x>>16); }
static void vector(float *v,int d,uint32_t id) {
 double norm=0;
 for(int j=0;j<d;j++){ v[j]=(float)((int)(mix(id*747796405U+(uint32_t)j*2891336453U)>>8)%2001-1000)/1000.0f; norm+=(double)v[j]*v[j]; }
 norm=sqrt(norm); if(norm==0) stop("zero vector");
 for(int j=0;j<d;j++)v[j]=(float)(v[j]/norm);
}
static int number(const char *s,int hi) { char *end; long v=strtol(s,&end,10); if(*end||v<1||v>hi)stop("invalid size");return (int)v; }
int main(int argc,char **argv) {
 int rank=0,ranks=1,threads=1;
#ifdef USE_MPI
 int provided;MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);
 if(provided<MPI_THREAD_FUNNELED)stop("MPI thread support too low");
 MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&ranks);
#endif
 if(argc!=4)stop("usage: search N D Q (N<=1000000,D<=512,Q<=128)");
 int n=number(argv[1],1000000),d=number(argv[2],512),nq=number(argv[3],128);
 if(ranks>n)stop("more ranks than vectors");
#ifdef _OPENMP
 #pragma omp parallel
 {
  #pragma omp single
  threads=omp_get_num_threads();
 }
#endif
 int begin=(int)((long long)n*rank/ranks),end=(int)((long long)n*(rank+1)/ranks),local=end-begin;
 float *x=malloc((size_t)local*d*sizeof(float)),*q=malloc((size_t)nq*d*sizeof(float));
 double *scores=malloc((size_t)local*sizeof(double)),*best=malloc((size_t)nq*sizeof(double));
 int *ids=malloc((size_t)nq*sizeof(int));
 double *gather_scores=malloc((size_t)ranks*sizeof(double));int *gather_ids=malloc((size_t)ranks*sizeof(int));
 if(!x||!q||!scores||!best||!ids||!gather_scores||!gather_ids)stop("allocation failed");
 for(int i=0;i<local;i++)vector(x+(size_t)i*d,d,(uint32_t)(begin+i+1));
 for(int k=0;k<nq;k++)vector(q+(size_t)k*d,d,(uint32_t)(10000001+k));
 /* One untimed dot-product pass warms memory/code. It does not initialize results. */
 volatile double warm=0; for(int j=0;j<d;j++)warm+=(double)x[j]*q[j];(void)warm;
#ifdef USE_MPI
 MPI_Barrier(MPI_COMM_WORLD);
#endif
 double start=now(),compute=0,selection=0,collective=0;
 for(int k=0;k<nq;k++) {
  double t=now();
  /* Each thread writes a different scores[i]; no shared best-index race. */
  #pragma omp parallel for schedule(static)
  for(int i=0;i<local;i++) {
   double dot=0;
   for(int j=0;j<d;j++)dot+=(double)x[(size_t)i*d+j]*q[(size_t)k*d+j];
   scores[i]=dot;
  }
  compute+=now()-t;t=now();
  int id=begin;double value=scores[0];
  for(int i=1;i<local;i++)if(scores[i]>value||(scores[i]==value&&begin+i<id)){value=scores[i];id=begin+i;}
  selection+=now()-t;t=now();
#ifdef USE_MPI
  /* All MPI calls are outside OpenMP regions: FUNNELED is sufficient. */
  MPI_Gather(&value,1,MPI_DOUBLE,gather_scores,1,MPI_DOUBLE,0,MPI_COMM_WORLD);
  MPI_Gather(&id,1,MPI_INT,gather_ids,1,MPI_INT,0,MPI_COMM_WORLD);
  if(rank==0)for(int r=0;r<ranks;r++)if(gather_scores[r]>value||(gather_scores[r]==value&&gather_ids[r]<id)){value=gather_scores[r];id=gather_ids[r];}
#endif
  if(ranks>1) collective+=now()-t;
  best[k]=value;ids[k]=id;
 }
 double elapsed=now()-start;
#ifdef USE_MPI
 double times[4]={elapsed,compute,selection,collective},maxima[4];
 MPI_Reduce(times,maxima,4,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);
 elapsed=maxima[0];compute=maxima[1];selection=maxima[2];collective=maxima[3];
#endif
 if(rank==0){
  printf("{\"n\":%d,\"d\":%d,\"queries\":%d,\"ranks\":%d,\"threads\":%d,\"seconds\":%.9g,\"compute_max_s\":%.9g,\"selection_max_s\":%.9g,\"collective_max_s\":%.9g,\"hits\":[",n,d,nq,ranks,threads,elapsed,compute,selection,collective);
  for(int k=0;k<nq;k++)printf("%s{\"id\":%d,\"score\":%.17g}",k?",":"",ids[k],best[k]);
  puts("]}");
 }
 free(x);free(q);free(scores);free(best);free(ids);free(gather_scores);free(gather_ids);
#ifdef USE_MPI
 MPI_Finalize();
#endif
 return 0;
}
"""
(WORK/'search.c').write_text(C_SOURCE)
print(C_SOURCE)


In [ ]:
# Independent tiny reference: 17 rows, 7 dimensions, 3 queries. No benchmark here.
def f32(v): return struct.unpack('f', struct.pack('f', v))[0]
def mix(x):
    x &= 0xffffffff
    x ^= x >> 16; x = (x * 0x7feb352d) & 0xffffffff
    x ^= x >> 15; x = (x * 0x846ca68b) & 0xffffffff
    return x ^ (x >> 16)
def vector(identifier, dimensions):
    v = [f32(f32((mix(identifier*747796405+j*2891336453) >> 8) % 2001-1000)/f32(1000))
         for j in range(dimensions)]
    norm = math.sqrt(sum(x*x for x in v))
    return [f32(x/norm) for x in v]
def expected_hits(n, d, queries):
    data = [vector(i+1, d) for i in range(n)]
    hits = []
    for k in range(queries):
        query = vector(10000001+k, d)
        scores = [sum(x*y for x,y in zip(row,query)) for row in data]
        best = max(range(n), key=lambda i: (scores[i], -i))
        hits.append(dict(id=best, score=scores[best]))
    return hits
if not shutil.which('gcc'):
    STATUS['E1_tiny_reference'] = 'NOT RUN'
    print('NOT RUN: gcc is missing in this kernel. Load the site environment and rerun here or in a compute terminal.')
else:
    run(['gcc','-O3','-std=c11','search.c','-lm','-o','search_serial'])
    actual = json.loads(run([WORK/'search_serial',17,7,3]))
    expected = expected_hits(17,7,3)
    assert len(actual['hits']) == len(expected)
    for a,b in zip(actual['hits'],expected):
        assert a['id']==b['id'] and math.isclose(a['score'],b['score'],abs_tol=1e-10,rel_tol=1e-10)
    STATUS['E1_tiny_reference'] = 'PASS'
    print('PASS: C agrees with the independent Python tiny reference.')
    print(json.dumps(actual['hits'],indent=2))


**Result and explanation:** the printed IDs and scores are actual tiny-run outputs from your environment.
The time field from this fixture is not a useful performance benchmark.
**Student experiment:** change only the tiny row count from 17 to 19 in both reference and C call.
**Takeaway:** deterministic inputs and an independent check make errors easier to isolate.

**Checkpoint 1:** why does running without a crash not prove search correctness?
A. Every parallel program is wrong. B. It may return the wrong item. C. More threads prove correctness. D. Slurm checks the mathematics.
<details><summary>Expected reasoning</summary>B. Completion is necessary, but neither the scheduler nor thread count checks the returned neighbour. A is false.</details>


## E2 · Threads over one collection

**Real problem:** every stored vector must be compared with each query.
**Prediction:** will four threads be four times faster? Which data can they share safely?
**Code:** `#pragma omp parallel for schedule(static)` splits row scoring. Each iteration writes its own
`scores[i]`; input vectors and the query are read-only. A serial scan then selects the local winner.
That scan is intentionally simple and can limit performance. Queries are processed sequentially in this guided version.

We compare a separately compiled serial baseline, OpenMP with one thread, then two and four threads.
Each configuration runs three times in one bounded allocation. We retain all repetitions and use medians.
Generation, normalization, allocation, compilation, launch and correctness comparison are outside the C timing region.
The timed region includes scoring, local selection and, for MPI, the per-query collective/merge.
This is **search-phase throughput**, not end-to-end request latency.


In [ ]:
FILES = {'environment_probe.sh': '#!/bin/bash -l\n# Read-only environment inspection; no jobs or packages are installed here.\necho "HOST=$(hostname)"\nmodule -t list 2>&1 || true\ntype -a gcc mpicc python3 apptainer 2>&1 || true\ngcc --version 2>&1 | head -1 || true\nmpicc --showme 2>&1 || true\napptainer --version 2>&1 || true\n# Compare compiler/MPI wrapper paths; do not mix system MPI and another stack.\n', 'container.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-replay\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=1\n#SBATCH --mem=256M\n#SBATCH --time=00:03:00\n#SBATCH --output=m5s5-replay-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nsource ./environment_probe.sh\nlscpu | head -25 || true\npython3 container_workflow.py\n', 'hybrid.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-hybrid\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-hybrid-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nsource ./environment_probe.sh\nlscpu | head -25 || true\npython3 run_experiments.py hybrid --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n', 'cpu.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-cpu\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-cpu-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nsource ./environment_probe.sh\nlscpu | head -25 || true\npython3 run_experiments.py cpu --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n', 'run_experiments.py': '"""Bounded guided experiments. Significant work requires a Slurm batch allocation.\n--local is for instructor validation on a local workstation, never a login node.\n"""\nfrom pathlib import Path\nimport argparse, hashlib, json, math, os, shutil, socket, subprocess, time, traceback\n\nROOT = Path(__file__).resolve().parent\n\ndef command(args, env=None, timeout=180):\n    p = subprocess.run([str(a) for a in args], text=True, capture_output=True,\n                       timeout=timeout, env=env)\n    if p.returncode:\n        raise RuntimeError(f\'{args!r}\\nexit={p.returncode}\\n{p.stdout}\\n{p.stderr}\')\n    return p.stdout\n\ndef equivalent(actual, reference):\n    assert (actual[\'n\'], actual[\'d\'], actual[\'queries\']) == (reference[\'n\'], reference[\'d\'], reference[\'queries\'])\n    assert len(actual[\'hits\']) == len(reference[\'hits\'])\n    for a, b in zip(actual[\'hits\'], reference[\'hits\']):\n        assert a[\'id\'] == b[\'id\'], (a, b)\n        assert math.isfinite(a[\'score\']) and math.isclose(a[\'score\'], b[\'score\'], rel_tol=1e-10, abs_tol=1e-10)\n    assert math.isfinite(actual[\'seconds\']) and actual[\'seconds\'] > 0\n\ndef main():\n    os.chdir(ROOT)\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'cpu\', \'hybrid\', \'container\'])\n    p.add_argument(\'--local\', action=\'store_true\')\n    p.add_argument(\'--n\', type=int, default=65536)\n    p.add_argument(\'--d\', type=int, default=128)\n    p.add_argument(\'--queries\', type=int, default=16)\n    a = p.parse_args()\n    if not a.local and not os.environ.get(\'SLURM_JOB_ID\'):\n        raise RuntimeError(\'NOT RUN: submit the supplied batch script; do not benchmark on a login/Jupyter node.\')\n    assert 1 <= a.n <= 1000000 and 1 <= a.d <= 512 and 1 <= a.queries <= 128\n    assert a.n*a.d*4 <= 256*1024*1024, \'Keep vector storage at or below 256 MiB for the 1 GB allocation\'\n    out = ROOT / \'results\'; out.mkdir(exist_ok=True)\n    label = f\'{a.mode}-{os.environ.get("SLURM_JOB_ID", "local")}-{time.time_ns()}\'\n    records = []\n    meta = dict(mode=a.mode, execution=\'LOCAL\' if a.local else \'SLURM\', job_id=os.environ.get(\'SLURM_JOB_ID\'),\n                host=socket.gethostname(), source_sha256=hashlib.sha256((ROOT/\'search.c\').read_bytes()).hexdigest(),\n                configuration=dict(n=a.n, d=a.d, queries=a.queries),\n                allocation={k: os.environ.get(k) for k in [\'SLURM_JOB_NUM_NODES\',\'SLURM_NTASKS\',\'SLURM_CPUS_PER_TASK\']})\n    try:\n        cc = shutil.which(\'gcc\')\n        if not cc: raise RuntimeError(\'gcc missing: load the site GCC module in site_env.sh\')\n        meta[\'compiler\'] = command([cc, \'--version\']).splitlines()[0]\n        meta[\'flags\'] = \'-O3 -std=c11; OpenMP adds -fopenmp; MPI adds -DUSE_MPI\'\n        command([cc, \'-O3\',\'-std=c11\',\'search.c\',\'-lm\',\'-o\',\'search_serial\'])\n        size = [str(a.n), str(a.d), str(a.queries)]\n        def run(exe, ranks=1, threads=1, prefix=None):\n            env = os.environ.copy()\n            env.update(OMP_NUM_THREADS=str(threads), OMP_DYNAMIC=\'FALSE\', OMP_PLACES=\'cores\', OMP_PROC_BIND=\'close\')\n            cmd = (prefix or []) + [str(exe)] + size\n            if not a.local:\n                cmd = [\'srun\',\'--exact\',\'--exclusive\',\'--nodes=1\',f\'--ntasks={ranks}\',f\'--cpus-per-task={threads}\',\'--cpu-bind=cores\'] + cmd\n            elif ranks > 1:\n                launcher = shutil.which(\'mpiexec\')\n                if not launcher: raise RuntimeError(\'local MPI launcher missing\')\n                cmd = [launcher, \'-n\', str(ranks)] + cmd\n            result = json.loads(command(cmd, env))\n            assert result[\'ranks\']==ranks and result[\'threads\']==threads, \'Runtime did not use the requested team size\'\n            return result, cmd\n        reference, _ = run(ROOT/\'search_serial\')\n        reference_name=label+\'-reference.json\'\n        (out/reference_name).write_text(json.dumps(reference,indent=2)+\'\\n\')\n        if a.mode == \'cpu\':\n            command([cc,\'-O3\',\'-std=c11\',\'-fopenmp\',\'search.c\',\'-lm\',\'-o\',\'search_omp\'])\n            configurations = [(\'serial\',ROOT/\'search_serial\',1,1,None)] + [(f\'omp-{t}\',ROOT/\'search_omp\',1,t,None) for t in [1,2,4]]\n        elif a.mode == \'hybrid\':\n            mpi = shutil.which(\'mpicc\')\n            if not mpi:\n                meta.update(status=\'SKIPPED\',reason=\'mpicc unavailable: load a site-compatible MPI module; CPU measurements remain usable\')\n                return\n            meta[\'mpi_wrapper\'] = command([mpi,\'--version\']).splitlines()[0]\n            command([mpi,\'-O3\',\'-std=c11\',\'-fopenmp\',\'-DUSE_MPI\',\'search.c\',\'-lm\',\'-o\',\'search_mpi\'])\n            configurations = [(\'mpi-2x2\',ROOT/\'search_mpi\',2,2,None)]\n        else:\n            image = Path(os.environ.get(\'M5S5_IMAGE\',\'/nonexistent\')).expanduser()\n            app = shutil.which(\'apptainer\')\n            if not app or not image.is_file():\n                meta.update(status=\'SKIPPED\',reason=\'Set M5S5_IMAGE to an existing trusted C/OpenMP-capable SIF and load Apptainer\')\n                return\n            meta[\'image_path\'] = str(image.resolve())\n            h = hashlib.sha256()\n            with image.open(\'rb\') as f:\n                for block in iter(lambda:f.read(1024*1024), b\'\'): h.update(block)\n            meta[\'image_sha256\'] = h.hexdigest()\n            prefix = [app,\'exec\',\'--cleanenv\',\'--bind\',str(ROOT)+\':/work\',\'--pwd\',\'/work\',str(image.resolve())]\n            # Compile inside the image, rather than assuming host libgomp/glibc compatibility.\n            build = prefix + [\'gcc\',\'-O3\',\'-std=c11\',\'-fopenmp\',\'/work/search.c\',\'-lm\',\'-o\',\'/work/search_container\']\n            if not a.local: build = [\'srun\',\'--exact\',\'-N1\',\'-n1\',\'-c1\'] + build\n            probe = subprocess.run(build,text=True,capture_output=True,timeout=60)\n            if probe.returncode:\n                meta.update(status=\'SKIPPED\',reason=\'Image lacks a compatible gcc/OpenMP build environment\',diagnostic=probe.stderr[-3000:])\n                return\n            os.environ[\'APPTAINERENV_OMP_NUM_THREADS\']=\'4\'\n            os.environ[\'APPTAINERENV_OMP_DYNAMIC\']=\'FALSE\'\n            os.environ[\'APPTAINERENV_OMP_PLACES\']=\'cores\'\n            os.environ[\'APPTAINERENV_OMP_PROC_BIND\']=\'close\'\n            configurations = [(\'container-omp-4\',\'/work/search_container\',1,4,prefix)]\n        for name, exe, ranks, threads, prefix in configurations:\n            for repeat in range(3):\n                result, cmd = run(exe,ranks,threads,prefix)\n                equivalent(result,reference)\n                record = dict(configuration=name,repeat=repeat+1,status=\'PASS\',command=cmd,**result)\n                records.append(record)\n                candidate_name=label+\'-\'+name+\'-\'+str(repeat+1)+\'.json\'\n                (out/candidate_name).write_text(json.dumps(record,indent=2)+\'\\n\')\n                print(json.dumps(record),flush=True)\n        if a.mode==\'cpu\':\n            manifest=dict(reference=\'results/\'+reference_name,candidate=\'results/\'+candidate_name,\n                          reference_sha256=hashlib.sha256((out/reference_name).read_bytes()).hexdigest(),\n                          candidate_sha256=hashlib.sha256((out/candidate_name).read_bytes()).hexdigest(),\n                          source_sha256=meta[\'source_sha256\'],cpu_job_id=meta[\'job_id\'],execution=meta[\'execution\'])\n            (ROOT/\'workflow_manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n        meta[\'status\']=\'PASS\'\n    except Exception as e:\n        meta.update(status=\'FAIL\',reason=str(e),traceback=traceback.format_exc())\n        raise\n    finally:\n        meta[\'records\']=records\n        (out/(label+\'.json\')).write_text(json.dumps(meta,indent=2)+\'\\n\')\n        print(\'RESULT\',meta[\'status\'],out/(label+\'.json\'),meta.get(\'reason\',\'\'),flush=True)\n\nif __name__==\'__main__\':main()\n', 'site_env.sh': '# SciTech configuration used in the successful instructor preflight, job 24943.\n# Apply inside batch jobs; do not edit shell startup files or the running kernel.\nexport PATH="/usr/bin:/bin:$PATH"\nunset LD_LIBRARY_PATH LIBRARY_PATH CPATH C_INCLUDE_PATH CPLUS_INCLUDE_PATH\nexport OMPI_CC=/usr/bin/gcc\n\n# Shared Python 3 image: no download, copy, installation or GPU is needed.\nexport M5S5_IMAGE=/data/software/containers/pytorch-2.8.0-cuda12.8.sif\n# Optional C-in-container work requires a separately verified C/OpenMP image.\n', 'container_workflow.py': '"""Guided native-C -> container-validation workflow inside a Slurm allocation.\nRequires a successful CPU run and an existing Python 3 SIF; no gcc in the image.\n"""\nfrom pathlib import Path\nimport hashlib, json, os, shutil, subprocess, sys, time\nfrom container_replay import sha\n\nROOT=Path(__file__).resolve().parent\n\ndef validate_manifest(root,manifest,require_slurm=True):\n    if require_slurm and manifest.get(\'execution\')!=\'SLURM\':\n        raise ValueError(\'Required CPU-stage evidence must come from Slurm, not a local debugging run\')\n    for key,hash_key in [(\'reference\',\'reference_sha256\'),(\'candidate\',\'candidate_sha256\')]:\n        if sha(root/manifest[key])!=manifest[hash_key]:\n            raise ValueError(\'Stale workflow input: \'+key+\'; rerun CPU stage\')\n    if sha(root/\'search.c\')!=manifest[\'source_sha256\']:\n        raise ValueError(\'Source changed: rebuild and rerun CPU stage\')\n\ndef main():\n    os.chdir(ROOT)\n    if not os.environ.get(\'SLURM_JOB_ID\'):\n        print(\'NOT RUN: submit container.sbatch; do not benchmark in this kernel.\');return 2\n    out=ROOT/\'replay\';out.mkdir(exist_ok=True)\n    label=f"job-{os.environ[\'SLURM_JOB_ID\']}-{time.time_ns()}"\n    report=dict(mode=\'container_replay\',execution=\'SLURM\',job_id=os.environ[\'SLURM_JOB_ID\'],\n                status=\'NOT RUN\',checks={})\n    def run(command,env=None,allow_failure=False):\n        p=subprocess.run([str(x) for x in command],text=True,capture_output=True,env=env,timeout=60)\n        with (out/(label+\'-commands.log\')).open(\'a\') as f:\n            f.write(json.dumps([str(x) for x in command])+\'\\n\'+p.stdout+p.stderr+f\'\\nexit={p.returncode}\\n\')\n        if p.returncode and not allow_failure:raise RuntimeError(p.stderr[-2000:] or p.stdout[-2000:])\n        return p\n    try:\n        manifest_path=ROOT/\'workflow_manifest.json\'\n        if not manifest_path.is_file():raise ValueError(\'Run cpu.sbatch first: workflow_manifest.json is missing\')\n        manifest=json.loads(manifest_path.read_text());report[\'manifest\']=manifest\n        validate_manifest(ROOT,manifest)\n        native=out/(label+\'-native.json\')\n        env=os.environ.copy();env[\'HPC_M5S5_HOST_SENTINEL\']=\'host-only\'\n        args=[\'--reference\',ROOT/manifest[\'reference\'],\'--candidate\',ROOT/manifest[\'candidate\']]\n        run([sys.executable,ROOT/\'container_replay.py\']+args+[\'--output\',native],env=env)\n        n=json.loads(native.read_text());report[\'checks\'][\'native_replay\']=\'PASS\'\n        image=os.environ.get(\'M5S5_IMAGE\')\n        if not shutil.which(\'apptainer\') or not image or not Path(image).expanduser().is_file():\n            report[\'reason\']=\'Required Apptainer step NOT RUN: load Apptainer and configure M5S5_IMAGE to an existing Python 3 SIF\'\n            report[\'checks\'][\'apptainer_replay\']=\'NOT RUN\';return 2\n        image=Path(image).expanduser().resolve();report[\'image\']=dict(path=str(image),sha256=sha(image))\n        prefix=[\'srun\',\'--exact\',\'-N1\',\'-n1\',\'-c1\',\'apptainer\',\'exec\',\'--cleanenv\',\n                \'--bind\',str(ROOT)+\':/work:ro\',\'--bind\',str(out)+\':/out\',\'--pwd\',\'/work\',str(image),\n                \'python3\',\'/work/container_replay.py\',\'--reference\',\'/work/\'+manifest[\'reference\'],\n                \'--candidate\',\'/work/\'+manifest[\'candidate\']]\n        # No GCC or host C runtime is needed in this Python validation container.\n        env.pop(\'APPTAINERENV_HPC_M5S5_HOST_SENTINEL\',None)\n        env.pop(\'SINGULARITYENV_HPC_M5S5_HOST_SENTINEL\',None)\n        clean_name=label+\'-container-clean.json\'\n        run(prefix+[\'--output\',\'/out/\'+clean_name],env=env)\n        c=json.loads((out/clean_name).read_text())\n        if c[\'result\']!=n[\'result\'] or c[\'inputs\']!=n[\'inputs\'] or c[\'provenance\'][\'script_sha256\']!=n[\'provenance\'][\'script_sha256\']:raise ValueError(\'Native/container scientific results or input hashes differ\')\n        report[\'checks\'][\'apptainer_replay\']=\'PASS\'\n        env[\'APPTAINERENV_HPC_M5S5_HOST_SENTINEL\']=\'explicitly-forwarded\'\n        forwarded_name=label+\'-container-forwarded.json\'\n        run(prefix+[\'--output\',\'/out/\'+forwarded_name],env=env)\n        f=json.loads((out/forwarded_name).read_text())\n        if f[\'provenance\'][\'marker\']!=\'explicitly-forwarded\' or f[\'result\']!=n[\'result\']:\n            raise ValueError(\'Environment forwarding/result check failed\')\n        report[\'checks\'][\'explicit_environment\']=\'PASS\'\n        report[\'observed_markers\']=dict(native=n[\'provenance\'][\'marker\'],clean=c[\'provenance\'][\'marker\'],forwarded=f[\'provenance\'][\'marker\'])\n        # A deliberately corrupted copy is rejected inside the container.\n        bad=json.loads((ROOT/manifest[\'candidate\']).read_text());bad[\'hits\'][0][\'score\']+=0.01\n        bad_name=label+\'-deliberately-bad.json\';(out/bad_name).write_text(json.dumps(bad))\n        bad_cmd=prefix.copy();bad_cmd[bad_cmd.index(\'--candidate\')+1]=\'/out/\'+bad_name\n        rejected_name=label+\'-rejected.json\'\n        p=run(bad_cmd+[\'--output\',\'/out/\'+rejected_name],env=env,allow_failure=True)\n        rejected=json.loads((out/rejected_name).read_text()) if (out/rejected_name).is_file() else {}\n        if p.returncode==0 or rejected.get(\'status\')!=\'FAIL\':raise ValueError(\'Corrupted output was not numerically rejected\')\n        report[\'checks\'][\'deliberate_error_rejected\']=\'PASS\'\n        report[\'status\']=\'PASS\'\n    except Exception as e:report.update(status=\'FAIL\',reason=str(e));return 1\n    finally:\n        (out/(label+\'-workflow.json\')).write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps(report,indent=2))\n        print(\'This validates native C outputs inside Apptainer; C/MPI execution inside the image is not claimed.\')\n    return 0\n\nif __name__==\'__main__\':sys.exit(main())\n', 'container_replay.py': '"""Portable result validation, used unchanged by native Python and a Python SIF.\nThis validates native C outputs; it does not claim C execution inside the image.\n"""\nfrom pathlib import Path\nimport argparse, hashlib, json, math, os, platform, sys\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1048576),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef require(condition,message):\n    if not condition:raise ValueError(message)\n\ndef compare(reference,candidate):\n    for key in [\'n\',\'d\',\'queries\']:\n        require(candidate.get(key)==reference.get(key),\'changed problem: \'+key)\n    topk=\'k\' in reference\n    require((\'k\' in candidate)==topk,\'changed result format\')\n    if topk:\n        for key in [\'k\',\'ties\']:require(candidate.get(key)==reference.get(key),\'changed problem: \'+key)\n    q=reference[\'queries\'];k=reference[\'k\'] if topk else 1\n    require(len(reference[\'hits\'])==q and len(candidate[\'hits\'])==q,\'missing queries\')\n    require(math.isfinite(candidate[\'seconds\']) and candidate[\'seconds\']>0,\'invalid recorded time\')\n    error=0.0\n    for expected,actual in zip(reference[\'hits\'],candidate[\'hits\']):\n        expected=expected if topk else [expected];actual=actual if topk else [actual]\n        require(len(expected)==k and len(actual)==k,\'wrong neighbour count\')\n        require(len({h[\'id\'] for h in actual})==k,\'duplicate neighbour ID\')\n        for position,(e,a) in enumerate(zip(expected,actual)):\n            require(type(a[\'id\']) is int and 0<=a[\'id\']<reference[\'n\'],\'invalid neighbour ID\')\n            require(a[\'id\']==e[\'id\'],\'ordered neighbour ID mismatch\')\n            require(math.isfinite(a[\'score\']) and math.isfinite(e[\'score\']),\'non-finite score\')\n            error=max(error,abs(a[\'score\']-e[\'score\']))\n            require(math.isclose(a[\'score\'],e[\'score\'],rel_tol=1e-10,abs_tol=1e-10),\'score mismatch\')\n            if position:\n                previous=actual[position-1]\n                require(previous[\'score\']>a[\'score\'] or (previous[\'score\']==a[\'score\'] and previous[\'id\']<a[\'id\']),\'incorrect tie/order rule\')\n    return dict(n=reference[\'n\'],d=reference[\'d\'],queries_checked=q,neighbours_per_query=k,\n                max_score_error=error,recorded_search_s=candidate[\'seconds\'],\n                recorded_queries_per_s=q/candidate[\'seconds\'])\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--reference\',required=True);p.add_argument(\'--candidate\',required=True)\n    p.add_argument(\'--output\',required=True)\n    args=p.parse_args()\n    record=dict(status=\'FAIL\',provenance=dict(python=sys.version.split()[0],executable=sys.executable,\n                 platform=platform.platform(),host=platform.node(),\n                 marker=os.environ.get(\'HPC_M5S5_HOST_SENTINEL\'),script_sha256=sha(__file__)))\n    try:\n        record[\'inputs\']=dict(reference_sha256=sha(args.reference),candidate_sha256=sha(args.candidate))\n        record[\'result\']=compare(json.loads(Path(args.reference).read_text()),json.loads(Path(args.candidate).read_text()))\n        record[\'status\']=\'PASS\'\n    except Exception as e:record[\'reason\']=str(e)\n    output=Path(args.output);output.parent.mkdir(parents=True,exist_ok=True)\n    output.write_text(json.dumps(record,indent=2)+\'\\n\')\n    print(json.dumps(record))\n    return 0 if record[\'status\']==\'PASS\' else 1\n\nif __name__==\'__main__\':sys.exit(main())\n', 'container_c.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-container-c\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-container-c-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nsource ./environment_probe.sh\nlscpu | head -25 || true\npython3 run_experiments.py container --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n'}
for name, text in FILES.items():
    target = WORK/name
    if name == 'site_env.sh' and target.exists():
        existing = target.read_text()
        legacy = existing.startswith('# Put the exact SciTech module load commands here') and not any(line.lstrip().startswith('export ') for line in existing.splitlines())
        if legacy:
            (WORK/'site_env.legacy-backup.sh').write_text(existing)
            print('Replacing the old comment-only environment template; backup saved.')
        else:
            print('Preserving existing site_env.sh; inspect it against the supplied configuration.')
            continue
    target.write_text(text)
print('Created:', ', '.join(FILES))
print('Batch environment:')
print((WORK/'site_env.sh').read_text())
print((WORK/'cpu.sbatch').read_text())
import shlex
print('SSH terminal: change to this exact notebook directory:')
print('cd ' + shlex.quote(str(WORK)))
print('Next: run the automatic submission/status cell below.')
print('Files prepared. Jobs NOT RUN until explicitly submitted.')


In [ ]:
# Run this cell to submit the guided jobs by default, or to refresh their status.
AUTO_RUN = True
WAIT_SECONDS = 45  # Includes queue wait; rerun this cell later if necessary.
import time

JOB_FILE = WORK/'guided_jobs.json'
JOB_STATES = {}
FAILED_STATES = {'FAILED','CANCELLED','TIMEOUT','OUT_OF_MEMORY','NODE_FAIL','BOOT_FAIL','DEADLINE','PREEMPTED'}
TRACKED_FILES = ['search.c','site_env.sh','run_experiments.py','container_workflow.py',
                 'container_replay.py','cpu.sbatch','hybrid.sbatch','container.sbatch']

def save_jobs(record):
    temporary = JOB_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(record,indent=2)+'\n')
    temporary.replace(JOB_FILE)

def scheduler_states(jobs):
    if not jobs:
        return {}
    ids = ','.join(jobs.values())
    states = {}
    if shutil.which('sacct'):
        q = subprocess.run(['sacct','-n','-X','-P','-j',ids,'--format=JobIDRaw,State'],
                           capture_output=True,text=True,timeout=10)
        if q.returncode == 0:
            for line in q.stdout.splitlines():
                fields = line.split('|')
                if len(fields)>1 and fields[0] in jobs.values():
                    states[fields[0]] = fields[1].split()[0].rstrip('+')
    if shutil.which('squeue'):
        q = subprocess.run(['squeue','--noheader','--jobs',ids,'--format=%i|%T|%r'],
                           capture_output=True,text=True,timeout=10)
        if q.returncode == 0:
            for line in q.stdout.splitlines():
                fields = line.strip().split('|')
                if len(fields)>1:
                    states[fields[0]] = fields[1]
    return {mode:states.get(job,'UNKNOWN') for mode,job in jobs.items()}

def pending_label(mode):
    state = JOB_STATES.get(mode)
    if state in FAILED_STATES:
        return 'FAIL (Slurm '+state+'; inspect the job log)'
    if state in {'PENDING','CONFIGURING','RUNNING','COMPLETING','SUSPENDED'}:
        return state
    if state == 'COMPLETED':
        return 'NOT RUN: Slurm completed but no matching current numerical report was loaded'
    return 'NOT RUN' if mode not in JOBS else 'SUBMITTED: scheduler status unavailable; inspect the job log'

JOBS = {}
if not shutil.which('sbatch'):
    print('NOT RUN: sbatch is unavailable here. Use the printed SSH working directory and batch scripts.')
elif not AUTO_RUN:
    print('Automatic submission disabled. Existing reports remain available below.')
else:
    if os.environ.get('M5S5_N','65536') != '65536' or os.environ.get('M5S5_QUERIES','16') != '16':
        raise RuntimeError('Unset M5S5_N/M5S5_QUERIES overrides before the default guided workflow.')
    fingerprint = hashlib.sha256(json.dumps({name:hashlib.sha256((WORK/name).read_bytes()).hexdigest()
                                            for name in TRACKED_FILES},sort_keys=True).encode()).hexdigest()
    record = json.loads(JOB_FILE.read_text()) if JOB_FILE.exists() else {'fingerprint':fingerprint,'jobs':{}}
    if record['fingerprint'] != fingerprint:
        raise RuntimeError('Code/configuration changed since submission. Do not reuse old results. Keep existing evidence and use a new work directory for a new experiment; inspect/cancel any old active jobs first.')
    JOBS = record['jobs']
    for mode in ['cpu','hybrid','container']:
        if mode in JOBS:
            print(mode,'reusing job',JOBS[mode],flush=True)
            continue
        command = ['sbatch','--parsable','--kill-on-invalid-dep=yes']
        if mode == 'hybrid':
            command += ['--dependency=afterok:'+JOBS['cpu']]
        elif mode == 'container':
            # CPU must pass. A hybrid failure must not prevent independent replay.
            command += ['--dependency=afterok:'+JOBS['cpu']+',afterany:'+JOBS['hybrid']]
        command += [mode+'.sbatch']
        response = subprocess.run(command,cwd=WORK,text=True,capture_output=True,timeout=20)
        if response.returncode:
            raise RuntimeError('Submission failed; already submitted job IDs were retained. '+response.stderr)
        job = response.stdout.strip().split(';')[0]
        if not job.isdigit():
            raise RuntimeError('Unexpected sbatch response: '+response.stdout)
        JOBS[mode] = job
        save_jobs(record)
        print(mode,'submitted job',job,flush=True)
    print('Submission is not PASS. Waiting briefly for scheduler completion...',flush=True)
    deadline = time.monotonic()+WAIT_SECONDS
    while True:
        JOB_STATES = scheduler_states(JOBS)
        active = any(s in {'PENDING','CONFIGURING','RUNNING','COMPLETING','SUSPENDED'} for s in JOB_STATES.values())
        if not active or time.monotonic() >= deadline:
            break
        time.sleep(min(3,max(0,deadline-time.monotonic())))
    for mode,job in JOBS.items():
        log_prefix = {'cpu':'m5s5-cpu','hybrid':'m5s5-hybrid','container':'m5s5-replay'}[mode]
        print(mode,'job',job,'Slurm:',JOB_STATES.get(mode,'UNKNOWN'),'log:',WORK/(log_prefix+'-'+job+'.out'))
    print('Now run the E4 report and E5 summary/plot cells. If jobs are still active, rerun this cell and those result cells later. Job IDs are reused; no automatic retries occur.')


**Default student route:** run the automatic submission/status cell above. It submits CPU → hybrid → container replay through Slurm and saves their IDs in `guided_jobs.json`. The replay requires CPU success but can still run after a failed MPI stage. Rerunning this cell reuses job IDs rather than creating duplicate jobs. Set `AUTO_RUN=False` only if you explicitly prefer manual submission.

**Queue wait:** the cell waits up to 45 seconds before returning control. PENDING/RUNNING is normal if the cluster is busy. Rerun the status cell and the E4/E5 result cells later to load completed evidence. A scheduler completion is not numerical PASS; the JSON checks determine that.

**SSH fallback:** if `sbatch` is unavailable in Jupyter, use the exact printed `cd` command, then `sbatch cpu.sbatch`; after CPU success submit `sbatch hybrid.sbatch` and `sbatch container.sbatch`. Inspect the job-specific logs and `sacct -j JOBID --format=JobID,State,ExitCode,Elapsed`. Do not submit manually as well as automatically for the same experiment.

**Student experiment:** predict the effect of Q=8 rather than Q=16, then use a separate work directory and submit `sbatch --export=ALL,M5S5_QUERIES=8 cpu.sbatch`. Keep its results separate from the default comparison. Do not overwrite source/configuration while jobs are active. To begin a new experiment preserve existing evidence and choose a new work directory. Cancel unwanted active jobs with `scancel JOBID`.


## E3 · MPI shards and threads inside each shard

**Real problem:** a larger collection may need to be partitioned across processes.
**Prediction:** if one rank finds the best item in its shard, is that necessarily the global answer?
**Code:** rank r owns `[N*r/P, N*(r+1)/P)`. Each rank stores only that interval plus the small query set.
Two `MPI_Gather` calls send one score and one ID per rank per query to rank 0, which merges the winners.
MPI calls happen outside threaded regions and request `MPI_THREAD_FUNNELED`.

**Run:** after confirming a compatible MPI toolchain and site launch configuration, submit `sbatch hybrid.sbatch`.
It uses **2 ranks × 2 threads = 4 CPUs**, on one node. Compare with the CPU job's 1 rank × 4 threads.
Both jobs request four CPUs; different allocations may land on different hardware, so compare host/CPU records.
This demonstrates a hybrid layout, not inter-node scaling. Multi-node work is not required here.

**Result and explanation:** each completed repetition checks every query's winning ID and score against
a fresh serial reference for the same input. Missing MPI is explicitly SKIPPED. A bad launcher or wrong answer is FAIL.
Record diagnostics; do not install a new MPI stack or guess process-manager options.
**Student experiment:** predict how many winner records rank 0 receives per query if ranks change from 2 to 4.
**Takeaway:** local correctness needs a correct global merge.

**Checkpoint 2:** which is a fair fixed-CPU comparison?
A. 4 ranks × 4 threads versus 1 × 4. B. Change dataset and rank count together.
C. 2 × 2 versus 1 × 4, same workload and comparable hardware. D. Compare queue waiting times.
<details><summary>Expected reasoning</summary>C. A changes the CPU budget, B changes work and D measures scheduling delay rather than search performance.</details>


## E4 · Native C computation and Apptainer validation

**Real problem:** a collaborator needs to rerun the analysis of your C search results in a known environment.
**Prediction:** can an existing Python image validate C results without containing a C compiler?
**Code / Run:** this core workflow uses an existing trusted **Python 3 SIF** available on SciTech.
It does not require gcc inside the image. The C computation remains native under Slurm; Python inside
Apptainer validates its actual JSON outputs. This does not claim that the C executable itself is portable.

After the CPU job passes, inspect `workflow_manifest.json`. It identifies the exact serial reference,
four-thread candidate, source hash, input hashes and job ID. Set the approved image path in `site_env.sh`:
`export M5S5_IMAGE=/data/software/containers/pytorch-2.8.0-cuda12.8.sif`, then submit `sbatch container.sbatch`.
This shared image executed Python 3.11.13 on a CPU allocation and passed the instructor replay checks in job 24943. Students use the shared file directly: no image download, private copy, package installation or GPU allocation is required. Do not use the CUDA development image for this Python activity.

The workflow performs these real steps:
1. Replay the same numerical validator with host Python.
2. Run that unchanged validator with `apptainer exec --cleanenv` and the existing Python SIF.
3. Bind the work folder **read-only at /work** and the output folder **read-write at /out**.
4. Compare scientific results and code/input hashes; record interpreter and image identities separately.
5. Inspect a host sentinel variable, then explicitly forward it using `APPTAINERENV_HPC_M5S5_HOST_SENTINEL`.
6. Corrupt a copied result deliberately and verify that the container validator rejects it.

**Result / explanation:** `replay/*-workflow.json` records each check. A missing image is **NOT RUN** for
this required activity, not automatic completion. Ask the teacher to resolve access and document any
exception. An incorrect numerical result is FAIL. Rejecting the deliberately bad copy is a successful
negative test. The SIF checksum establishes identity, not publisher trust. Host architecture/kernel and
resource configuration still matter. `--cleanenv` reduces inherited environment variables; inspect the
recorded values rather than assuming the image has no environment of its own.

**Student experiment:** copy one recorded candidate, alter one score and explain the rejection. Then
change `search.c` without rerunning the CPU stage: explain why the workflow refuses stale provenance.
Restore the source or rerun the producing stage before validating new results.
**Takeaway:** computation, portable analysis and an auditable handover form one reproducible workflow.

**Optional extension:** `container_c.sbatch` runs the earlier C/OpenMP-in-image experiment using a
verified gcc/OpenMP image. MPI-in-container and GPUs need separate compatibility validation and are
not required. Spack installation and workflow-framework syntax are not required; explain how modules,
package specifications and images preserve different parts of an environment.


In [ ]:
workflow_files=sorted((WORK/'replay').glob('*-workflow.json'),key=lambda p:p.stat().st_mtime_ns) if (WORK/'replay').exists() else []
if workflow_files:
    workflow=json.loads(workflow_files[-1].read_text())
    current_manifest=json.loads((WORK/'workflow_manifest.json').read_text()) if (WORK/'workflow_manifest.json').is_file() else None
    fresh=workflow.get('manifest')==current_manifest and current_manifest is not None and current_manifest.get('source_sha256')==hashlib.sha256((WORK/'search.c').read_bytes()).hexdigest()
    STATUS['Apptainer_workflow']=workflow['status'] if fresh else 'NOT RUN'
    print(json.dumps(workflow,indent=2))
    if not fresh: print('NOT RUN for current inputs: report is stale; rerun the producing/validation stages.')
    print('C computation was native; this report covers containerized result validation.')
else:
    STATUS['Apptainer_workflow']=pending_label('container') if 'pending_label' in globals() else 'NOT RUN'
    print('Apptainer workflow:',STATUS['Apptainer_workflow'])
    print('If a job is active, rerun the submission/status cell and this report cell after completion.')


## E5 · Read the evidence, not an expected speedup

**Real problem:** timings from different problem sizes or failed jobs cannot be mixed.
**Prediction:** if four threads are slower, has the experiment necessarily failed?
**Code / Run:** load the latest report per mode for the default 65536 × 128 collection and 16 queries,
with the current source hash. All tables below use actual records from your files. There is no fallback timing dataset.
If a job is still queued or failed before producing a report, inspect its scheduler state and log.


In [ ]:
DEFAULT = dict(n=65536,d=128,queries=16)
source_hash = hashlib.sha256((WORK/'search.c').read_bytes()).hexdigest()
reports = {}
for path in sorted((WORK/'results').glob('*.json'), key=lambda p:p.stat().st_mtime_ns) if (WORK/'results').exists() else []:
    r = json.loads(path.read_text())
    if r.get('source_sha256')==source_hash and r.get('configuration')==DEFAULT:
        reports[r['mode']] = r
summary=[]
for mode in ['cpu','hybrid']:
    report=reports.get(mode)
    STATUS[mode] = report['status'] if report else (pending_label(mode) if 'pending_label' in globals() else 'NOT RUN')
    print(mode, STATUS[mode], report.get('reason','') if report else '')
    if not report or report['status']!='PASS': continue
    print('Evidence:',report['execution'],'job',report['job_id'],'host',report['host'])
    groups={}
    for row in report['records']: groups.setdefault(row['configuration'],[]).append(row)
    for name, values in groups.items():
        assert len(values)==3 and all(v['status']=='PASS' for v in values)
        t=median(v['seconds'] for v in values)
        summary.append(dict(configuration=name,median_s=t,min_s=min(v['seconds'] for v in values),
                            max_s=max(v['seconds'] for v in values),queries_per_s=DEFAULT['queries']/t,
                            cpus=values[0]['ranks']*values[0]['threads'],execution=report['execution'],
                            job_id=report['job_id'],host=report['host']))
baseline=next((s['median_s'] for s in summary if s['configuration']=='serial'),None)
for row in summary:
    row['speedup_vs_serial']=baseline/row['median_s'] if baseline else None
    omp1=next((v['median_s'] for v in summary if v['configuration']=='omp-1'),None)
    row['omp_speedup_vs_omp1']=omp1/row['median_s'] if omp1 and row['configuration'].startswith('omp-') else None
    row['omp_efficiency']=row['omp_speedup_vs_omp1']/row['cpus'] if row['omp_speedup_vs_omp1'] is not None else None
    print(row)
if summary:
    with (WORK/'summary.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
if 'Apptainer_workflow' in STATUS:
    STATUS['container']=STATUS['Apptainer_workflow']
    print('Required Apptainer replay:',STATUS['container'])
if not summary: print('No matching completed measurements yet. Refresh status/results after the jobs finish; no performance claim is made.')


In [ ]:
# A chart is available even when matplotlib is not installed.
if summary:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        from html import escape
        width,height,left,top,plot_width,row_height=980,440,140,55,650,62
        maximum=max(s['max_s'] for s in summary)*1000*1.12
        elements=['<svg xmlns="http://www.w3.org/2000/svg" width="980" height="440" viewBox="0 0 980 440">',
                  '<rect width="980" height="440" fill="white"/>',
                  '<g font-family="sans-serif" font-size="17" fill="#222">',
                  '<text x="140" y="28">Measured search time (ms): median and min–max of three runs</text>']
        for i,row in enumerate(summary):
            y=top+i*row_height
            low=row['min_s']*1000/maximum*plot_width
            high=row['max_s']*1000/maximum*plot_width
            middle=row['median_s']*1000/maximum*plot_width
            label=escape(row['configuration'])
            elements.extend([f'<text x="125" y="{y+23}" text-anchor="end">{label}</text>',
                f'<rect x="{left}" y="{y}" width="{middle:.2f}" height="32" fill="#2F5597"/>',
                f'<path d="M {left+low:.2f} {y+16} H {left+high:.2f} M {left+low:.2f} {y+8} V {y+24} M {left+high:.2f} {y+8} V {y+24}" stroke="#111" fill="none" stroke-width="2"/>',
                f'<text x="{left+high+12:.2f}" y="{y+23}">{row["median_s"]*1000:.2f} ms</text>'])
        elements.extend(['<text x="140" y="405">Actual Slurm records only. Short runs and one node limit generalisation.</text>','</g></svg>'])
        svg=''.join(elements)
        chart_path=WORK/'measured_times.svg';chart_path.write_text(svg)
        try:
            from IPython.display import SVG,display
            display(SVG(svg))
        except ImportError:
            print('Chart saved:',chart_path)
        print('Standard-library SVG chart saved:',chart_path)
    else:
        labels=[s['configuration'] for s in summary]
        medians=[s['median_s']*1000 for s in summary]
        errors=[[1000*(s['median_s']-s['min_s']) for s in summary],
                [1000*(s['max_s']-s['median_s']) for s in summary]]
        fig,ax=plt.subplots(figsize=(8,4))
        ax.bar(labels,medians,yerr=errors,capsize=4,color='#2F5597')
        ax.set_ylabel('Median search time (ms), min–max of three runs')
        ax.tick_params(axis='x',rotation=20);fig.tight_layout()
        fig.savefig(WORK/'measured_times.png',dpi=160);plt.show()
else:
    print('No chart yet: wait for completed jobs, then rerun the summary and chart cells.')


**Result / explanation:** use the range of repetitions as well as the median. A short search may not
show useful scaling. Thread overhead, a serial selection scan, memory traffic and repeated collectives
are hypotheses, not conclusions until supported by measurements.

The raw records contain `compute_max_s`, `selection_max_s` and `collective_max_s`.
For MPI each is a maximum across ranks; these maxima may belong to different ranks and **must not
be added as a mutually exclusive runtime breakdown**. Collective time includes waiting and root merge,
not just network transfer. Allocation, initialization and input generation are outside these phase timings.

**Student experiment:** inspect one raw CPU and hybrid report. Propose one change to test the dominant
cost while keeping the input and correctness rule fixed. Do not claim memory-bandwidth saturation solely
because scaling flattened. A larger query count is a sensitivity experiment, not fixed-workload speedup.
**Takeaway:** a defensible result states what ran, how it was checked and what the measurements support.

**Checkpoint 3:** the container returns identical neighbours but is slower. Best conclusion?
A. Containers always slow applications. B. The result is invalid. C. Record this result and investigate
compiler/runtime/placement differences. D. Replace the measured timing with the native one.
<details><summary>Expected reasoning</summary>C. One comparison does not establish a universal performance effect. Numerical agreement and speed are different questions.</details>

## Exit and handover
Keep `search.c`, `run_experiments.py`, `site_env.sh`, batch files, source/image hashes, raw JSON, Slurm
logs and summary CSV. Record the repository commit used to obtain this notebook and exact commands.
Record any local edits with their own source hash. Never label workstation results as SciTech measurements.

The separate individual assignment requires **batched top-k retrieval**, an independently justified
optimization and your own evidence. The group milestone uses your group's chosen project, not this
guided output. Course assessment and AI-use rules continue to apply.

### Instructor validation boundary
Instructor-provided SciTech job **24943**, host **haskell**, completed with all attempted checks PASS: system GCC 14.2.0 build; independent baseline correctness fixtures; checker rejection tests; Q=32/Q=64 assignment baseline workloads; guided OpenMP teams 1/2/4; guided MPI 2 ranks × 2 threads; and Apptainer replay with read-only/writable binds, explicit environment forwarding and deliberate bad-result rejection. The shared image was `/data/software/containers/pytorch-2.8.0-cuda12.8.sif`.

This preflight validates supplied code and capabilities, not a student's assessed top-k implementation. It also does not establish that this notebook's complete prepare → submit → read reports → plot sequence has been executed on SciTech. Your own CPU/hybrid/replay stages remain NOT RUN until their current matching reports exist. Published outputs are cleared. No instructor timing is substituted for student measurements.

Primary teaching references: official syllabus Sessions 21–25 (pp.15–17); repository cluster guide;
M5S3 environment/container notebook; M4S5 asynchronous notebook conventions. No new theory reading is required.

The automatic submission/status control was checked locally with simulated Slurm responses for dependencies, repeat execution, partial submission failures and changed configuration. Its actual SciTech scheduling and the remaining notebook MPI/replay stages still require fresh execution evidence. Notebook CPU job 24944 completed on SciTech with checked serial/OpenMP records.

Instructor export confirms automatic notebook stages on SciTech: CPU 24945 PASS, hybrid 24946 PASS, and required replay 24947 PASS, all on haskell. Its matplotlib chart was unavailable; the new standard-library SVG fallback was validated locally against those recorded timings. The chart change does not alter cluster computation or submission.
